In [1]:
#IMPORTS
from pynq import Overlay, allocate, PL, MMIO
from pynq.lib.video import VideoMode
import ipywidgets as widgets
from IPython.display import display
import cv2
import numpy as np
from enum import Enum
import time
import os

projectDir = "/home/xilinx/jupyter_notebooks/PynqZ2-ORB-Accelerator"
PL.reset()
overlay = Overlay(projectDir + "/VivadoHW.bit")
overlay.download()

In [2]:
#SETUP AND DEFINES
DEFAULT_WIDTH, DEFAULT_HEIGHT = 1280, 720
IMAGES_DIR = projectDir + "/images"
VALID_IMAGE_EXTENSIONS = ('.jpg', '.jpeg', '.png', '.bmp')
available_images = [os.path.join(IMAGES_DIR, f) for f in os.listdir(IMAGES_DIR) if f.lower().endswith(VALID_IMAGE_EXTENSIONS)] if os.path.exists(IMAGES_DIR) else []
print(overlay.ip_dict.keys())
tpg = overlay.v_tpg_0
vdma = overlay.axi_vdma_0
vdma.writechannel.mode = VideoMode(DEFAULT_WIDTH, DEFAULT_HEIGHT, 24)
buttons = overlay.axi_gpios.axi_gpio_0
imageResolution = overlay.axi_gpios.axi_gpio_1
image_widget = widgets.Image(format='jpeg', width=DEFAULT_WIDTH, height=DEFAULT_HEIGHT)
stream_switch = overlay.orb_pipeline.axis_switch_0
current_ps_frame = None
video_cap = None
current_ps_frame = None
from enum import Enum

class VideoSource(Enum): # XX -> (videoSource id, axis_switch id)
    HDMI = (0, 0)
    TPG_COLOR_BAR_BOX = (1, 1)
    TPG_GRID_BOX = (2, 1)
    PS_TO_PL = (3, 2)

ps_to_pl_selected_image = 0
selected_source = None

dict_keys(['axi_gpios/axi_gpio_1', 'axi_gpios/axi_gpio_0', 'axi_intc_0', 'axi_vdma_0', 'v_tpg_0', 'orb_pipeline/axis_switch_0', 'processing_system7_0'])


In [ ]:
def setupVideoSource(new_src: VideoSource):
    global selected_source, ps_to_pl_selected_image, current_ps_frame  
    if selected_source == new_src and new_src != VideoSource.PS_TO_PL:
        return

    was_reading = vdma.readchannel.running
    if was_reading:
        vdma.readchannel.stop()
    if vdma.writechannel.running:
        vdma.writechannel.stop()

    selected_source = new_src

    imageResolution.channel1.write(DEFAULT_HEIGHT, 0xFFFFFFFF)
    imageResolution.channel2.write(DEFAULT_WIDTH, 0xFFFFFFFF)
    
    tpg.write(0x0000, 0x0)
    tpg.write(0x0010, DEFAULT_HEIGHT)
    tpg.write(0x0018, DEFAULT_WIDTH)

    if selected_source == VideoSource.TPG_COLOR_BAR_BOX:
        tpg.write(0x0020, 9); tpg.write(0x0028, 1); tpg.write(0x0038, 8)
        tpg.write(0x00D8, 0); tpg.write(0x0058, 150); tpg.write(0x0078, 200)
        tpg.write(0x0080, 200); tpg.write(0x0088, 50); tpg.write(0x0090, 150)
        tpg.write(0x0000, 0x81)
    elif selected_source == VideoSource.TPG_GRID_BOX:
        tpg.write(0x0020, 11); tpg.write(0x0028, 1); tpg.write(0x0038, 8)
        tpg.write(0x00D8, 0); tpg.write(0x0058, 150); tpg.write(0x0078, 200)
        tpg.write(0x0080, 200); tpg.write(0x0088, 50); tpg.write(0x0090, 150)
        tpg.write(0x0000, 0x81)

    stream_switch.write(0x040, selected_source.value[1])
    stream_switch.write(0x000, 0x2)

    if selected_source == VideoSource.PS_TO_PL:
        if available_images:
            current_image_path = available_images[ps_to_pl_selected_image]
            print(f"Loading image [{ps_to_pl_selected_image + 1}/{len(available_images)}] for PS_TO_PL: {current_image_path}")
            
            img = cv2.imread(current_image_path)
            img = cv2.resize(img, (DEFAULT_WIDTH, DEFAULT_HEIGHT))
            current_ps_frame = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            ps_to_pl_selected_image = (ps_to_pl_selected_image + 1) % len(available_images)
            vdma.writechannel.start()
        else:
            print("No images available to load.")
    
    if was_reading:
        vdma.readchannel.start()
    print("done")


vdma.readchannel.mode = VideoMode(DEFAULT_WIDTH, DEFAULT_HEIGHT, 8)
setupVideoSource(VideoSource.TPG_COLOR_BAR_BOX)
vdma.readchannel.start()

display(image_widget)

print("Streaming via VDMA interrupts...")
old_button_value = buttons.read(0)

try:
    while True:
        if selected_source == VideoSource.PS_TO_PL and current_ps_frame is not None:
            buf = vdma.writechannel.newframe()
            np.copyto(buf, current_ps_frame)
            vdma.writechannel.writeframe(buf)

        # READ AND DISPLAY FRAME
        frame = vdma.readchannel.readframe()
        _, jpeg = cv2.imencode('.jpg', frame)
        image_widget.value = jpeg.tobytes()
        
        # DETECT BUTTON AND SWITCH VIDEO SOURCE
        button_value = buttons.read(0)
        if button_value != old_button_value and button_value > 0:
            if button_value & 0x1:
                setupVideoSource(VideoSource.HDMI)
            elif button_value & 0x2:
                setupVideoSource(VideoSource.TPG_COLOR_BAR_BOX)
            elif button_value & 0x4:
                setupVideoSource(VideoSource.TPG_GRID_BOX)
            elif button_value & 0x8:
                setupVideoSource(VideoSource.PS_TO_PL)
                printNum = 0
        
        old_button_value = button_value
        

except KeyboardInterrupt:
    print("\nStopping stream...")
finally:
    tpg.write(0x00, 0x0)
    vdma.readchannel.stop()
    print("Stream stopped cleanly.")

done


Image(value=b'', format='jpeg', height='720', width='1280')

Streaming via VDMA interrupts...
Loading image [1/2] for PS_TO_PL: /home/xilinx/jupyter_notebooks/PynqZ2-ORB-Accelerator/images/park_and_coke.jpg
done
writechannel running: True
readchannel running: True
buf shape: (720, 1280, 3) buf max: 255
writechannel running: True
readchannel running: True
buf shape: (720, 1280, 3) buf max: 255
writechannel running: True
readchannel running: True
buf shape: (720, 1280, 3) buf max: 255
writechannel running: True
readchannel running: True
buf shape: (720, 1280, 3) buf max: 255
writechannel running: True
readchannel running: True
buf shape: (720, 1280, 3) buf max: 255
writechannel running: True
readchannel running: True
buf shape: (720, 1280, 3) buf max: 255
writechannel running: True
readchannel running: True
buf shape: (720, 1280, 3) buf max: 255
writechannel running: True
readchannel running: True
buf shape: (720, 1280, 3) buf max: 255
writechannel running: True
readchannel running: True
buf shape: (720, 1280, 3) buf max: 255
writechannel running: